IMPORTING NECESSARY LIBRARIES

In [2]:
import pandas as pd
import unicodedata
import re
import importlib, sys, subprocess
from pathlib import Path


LOADING DATASETS

In [3]:
path = "/content/drive/MyDrive/AutoEIT Test Files/Sample Audio Files and Transcriptions/AutoEIT Sample Transcriptions for Scoring.xlsx"
xls = pd.ExcelFile(path)

xls.sheet_names


['Info', '38001-1A', '38002-2A', '38004-2A', '38006-2A']

In [4]:
df = pd.read_excel(path, sheet_name="38001-1A")
df.head()


,Sentence,Stimulus,Transcription Rater 1,Score
0,1,Quiero cortarme el pelo (7),Quiero cortarme el pelo,NaN
1,2,El libro está en la mesa (7),El libro está en la mesa,NaN
2,3,El carro lo tiene Pedro (8),El carro lo tiene Pedro,NaN
3,4,El se ducha cada mañana (9),El se ducha cada mañana,NaN
4,5,¿Qué dice usted que va a hacer hoy? (9),Que dices ustedes se que van a hacer hoy?,NaN


In [5]:
# Recreate only Stimulus_clean and Transcription_clean (start fresh)

# --- Edit these if needed ---
path = "/content/drive/MyDrive/AutoEIT Test Files/Sample Audio Files and Transcriptions/AutoEIT Sample Transcriptions for Scoring.xlsx"
sheet_name = "38001-1A"

# load sheet
xls = pd.ExcelFile(path)
if sheet_name not in xls.sheet_names:
    raise ValueError(f"Sheet '{sheet_name}' not found. Available sheets: {xls.sheet_names}")
df = pd.read_excel(path, sheet_name=sheet_name)

# normalization function
def normalize_text(s, remove_parenthetical=False):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    if remove_parenthetical:
        s = re.sub(r"\s*\(\d+\)\s*$", "", s)  # remove trailing parenthetical counts like " (7)"
    s = s.lower()
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")  # remove accents
    s = re.sub(r"[^\w\s]", " ", s)  # remove punctuation (keep words and spaces)
    s = re.sub(r"\s+", " ", s).strip()  # collapse multiple spaces
    return s

# remove any existing cleaned columns to avoid duplicates
for col in ["Stimulus_clean", "Transcription_clean"]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# determine source columns robustly
stim_src_candidates = ["Stimulus", "Stimulus_clean_lower", "Stimulus_clean"]
trans_src_candidates = ["Transcription Rater 1", "Transcription", "Transcription_clean"]

stim_src = next((c for c in stim_src_candidates if c in df.columns), None)
trans_src = next((c for c in trans_src_candidates if c in df.columns), None)

if stim_src is None:
    raise KeyError("No stimulus source column found. Expected one of: " + ", ".join(stim_src_candidates))
if trans_src is None:
    raise KeyError("No transcription source column found. Expected one of: " + ", ".join(trans_src_candidates))

# create canonical cleaned columns (overwrite if existed)
df["Stimulus_clean"] = df[stim_src].apply(lambda s: normalize_text(s, remove_parenthetical=True))
df["Transcription_clean"] = df[trans_src].apply(lambda s: normalize_text(s, remove_parenthetical=False))

# produce tidy DataFrame with only the requested columns
comparison_df = df[["Sentence", "Stimulus_clean", "Transcription_clean"]].copy()

# preview
print("Preview of cleaned data (first 20 rows):")
display(comparison_df.head(20))


Preview of cleaned data (first 20 rows):


,Sentence,Stimulus_clean,Transcription_clean
0,1,quiero cortarme el pelo,quiero cortarme el pelo
1,2,el libro esta en la mesa,el libro esta en la mesa
2,3,el carro lo tiene pedro,el carro lo tiene pedro
3,4,el se ducha cada manana,el se ducha cada manana
4,5,que dice usted que va a hacer hoy,que dices ustedes se que van a hacer hoy
5,6,dudo que sepa manejar muy bien,dudo que sepa manajar bien
6,7,las calles de esta ciudad son muy anchas,las calles de esta cuidad son muy anchas
7,8,puede que llueva manana todo el dia,puede que lleva manana todo el dia
8,9,las casas son muy bonitas pero caras,las casas son muy bonitas pero muy cadas
9,10,me gustan las peliculas que acaban bien,me gustan las peliculas que acaban bien


In [6]:
# Reset Score_manual and assign Score 4 for exact cleaned-string equality only


# ensure rapidfuzz is available for later steps (no use here for assignment)
pkg = "rapidfuzz"
if importlib.util.find_spec(pkg) is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])
from rapidfuzz import fuzz

# operate on the working DataFrame (comparison_df must already exist)
df = comparison_df.copy()

# ensure cleaned columns exist
if "Stimulus_clean" not in df.columns or "Transcription_clean" not in df.columns:
    raise KeyError("Expected 'Stimulus_clean' and 'Transcription_clean' in comparison_df")

# reset Score_manual
df["Score_manual"] = pd.NA

# compute token_ratio and char_ratio for inspection (0-1)
df["token_ratio"] = df.apply(
    lambda r: fuzz.token_set_ratio(str(r["Stimulus_clean"]), str(r["Transcription_clean"])) / 100.0,
    axis=1
)
df["char_ratio"] = df.apply(
    lambda r: fuzz.ratio(str(r["Stimulus_clean"]), str(r["Transcription_clean"])) / 100.0,
    axis=1
)

# assign Score 4 only for exact cleaned-string equality
mask_4 = df["Stimulus_clean"] == df["Transcription_clean"]
df.loc[mask_4, "Score_manual"] = 4

# show rows assigned 4
print("Rows assigned Score 4 (exact cleaned-string equality):")
display(df.loc[mask_4, ["Sentence", "Stimulus_clean", "Transcription_clean", "token_ratio", "char_ratio", "Score_manual"]])

# show rows not assigned yet
print("\nRows NOT assigned yet (for next steps):")
display(df.loc[~mask_4, ["Sentence", "Stimulus_clean", "Transcription_clean", "token_ratio", "char_ratio"]].reset_index(drop=True))

# persist back
comparison_df = df.copy()


Rows assigned Score 4 (exact cleaned-string equality):


,Sentence,Stimulus_clean,Transcription_clean,token_ratio,char_ratio,Score_manual
0,1,quiero cortarme el pelo,quiero cortarme el pelo,1.0,1.0,4
1,2,el libro esta en la mesa,el libro esta en la mesa,1.0,1.0,4
2,3,el carro lo tiene pedro,el carro lo tiene pedro,1.0,1.0,4
3,4,el se ducha cada manana,el se ducha cada manana,1.0,1.0,4
9,10,me gustan las peliculas que acaban bien,me gustan las peliculas que acaban bien,1.0,1.0,4
11,12,despues de cenar me fui a dormir tranquilo,despues de cenar me fui a dormir tranquilo,1.0,1.0,4
14,15,ella solo bebe cerveza y no come nada,ella solo bebe cerveza y no come nada,1.0,1.0,4
26,27,le pedi a un amigo que me ayudara con la tarea,le pedi a un amigo que me ayudara con la tarea,1.0,1.0,4



Rows NOT assigned yet (for next steps):


,Sentence,Stimulus_clean,Transcription_clean,token_ratio,char_ratio
0,5,que dice usted que va a hacer hoy,que dices ustedes se que van a hacer hoy,0.892308,0.904110
1,6,dudo que sepa manejar muy bien,dudo que sepa manajar bien,0.892857,0.892857
2,7,las calles de esta ciudad son muy anchas,las calles de esta cuidad son muy anchas,0.975000,0.975000
3,8,puede que llueva manana todo el dia,puede que lleva manana todo el dia,0.985507,0.985507
4,9,las casas son muy bonitas pero caras,las casas son muy bonitas pero muy cadas,0.972222,0.921053
5,11,el chico con el que yo salgo es espanol,el chico con yo salgo um esta bien,0.828571,0.739726
6,13,quiero una casa en la que vivan mis animales,quiero una casa que vivan los animales,0.944444,0.878049
7,14,a nosotros nos fascinan las fiestas grandiosas,a nosotros nos fascina los fiestas grandiosos,0.945055,0.945055
8,16,me gustaria que el precio de las casas bajara,me gustaria que las precios de las casas bajaron,0.921348,0.924731
9,17,cruza a la derecha y despues sigue todo recto,cruza a la derecha y sigue a la izquierda,0.838710,0.674419


In [7]:
# Assign Score 3 (preserve existing Score_manual values)
# Rules:
# - Do not overwrite existing scores.
# - Assign 3 when:
#   a) token_ratio == 1.0 AND char_ratio >= char_threshold_for_3 AND cleaned strings are not exactly equal
#   b) OR token_ratio >= token_threshold_for_3 and token_ratio < 1.0 (cleaned strings not exactly equal)
# - Compute token_ratio and char_ratio if missing.
# - Print rows newly assigned 3 and remaining NA rows for review.

pkg = "rapidfuzz"
if importlib.util.find_spec(pkg) is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])
from rapidfuzz import fuzz

# --- Configurable thresholds ---
char_threshold_for_3 = 0.95   # char_ratio threshold to promote token==1.0 non-exact -> 3
token_threshold_for_3 = 0.95  # token_ratio >= this and <1.0 -> 3
# -------------------------------

# operate on the working DataFrame
df = comparison_df.copy()

# sanity checks
if "Stimulus_clean" not in df.columns or "Transcription_clean" not in df.columns:
    raise KeyError("Expected 'Stimulus_clean' and 'Transcription_clean' in comparison_df")

# ensure Score_manual exists
if "Score_manual" not in df.columns:
    df["Score_manual"] = pd.NA

# compute token_ratio and char_ratio if missing (0-1)
if "token_ratio" not in df.columns:
    df["token_ratio"] = df.apply(
        lambda r: fuzz.token_set_ratio(str(r["Stimulus_clean"]), str(r["Transcription_clean"])) / 100.0,
        axis=1
    )
if "char_ratio" not in df.columns:
    df["char_ratio"] = df.apply(
        lambda r: fuzz.ratio(str(r["Stimulus_clean"]), str(r["Transcription_clean"])) / 100.0,
        axis=1
    )

# mask for rows still unscored (do not overwrite existing 4/3/2/1)
mask_na = df["Score_manual"].isna()

# token==1.0 but not exact and high char similarity
mask_token1_char_high = mask_na & (df["token_ratio"] == 1.0) & (df["char_ratio"] >= char_threshold_for_3) & (df["Stimulus_clean"] != df["Transcription_clean"])

# token high band (>= token_threshold_for_3 and < 1.0) and not exact
mask_token_high = mask_na & (df["token_ratio"] >= token_threshold_for_3) & (df["token_ratio"] < 1.0) & (df["Stimulus_clean"] != df["Transcription_clean"])

mask_assign_3 = mask_token1_char_high | mask_token_high

# Apply assignment
df.loc[mask_assign_3, "Score_manual"] = 3

# Reporting
print("Rows newly assigned Score_manual = 3:")
if mask_assign_3.any():
    display(df.loc[mask_assign_3, ["Sentence","Stimulus_clean","Transcription_clean","token_ratio","char_ratio","Score_manual"]])
else:
    print("(none)")

print("\nRows still unscored (NA) after this pass — for next steps or manual review:")
if df["Score_manual"].isna().any():
    display(df.loc[df["Score_manual"].isna(), ["Sentence","Stimulus_clean","Transcription_clean","token_ratio","char_ratio"]])
else:
    print("(none)")

print("\nCounts by Score_manual after this pass:")
print(df["Score_manual"].value_counts(dropna=False).sort_index())

# persist back
comparison_df = df.copy()


Rows newly assigned Score_manual = 3:


,Sentence,Stimulus_clean,Transcription_clean,token_ratio,char_ratio,Score_manual
6,7,las calles de esta ciudad son muy anchas,las calles de esta cuidad son muy anchas,0.975000,0.975000,3
7,8,puede que llueva manana todo el dia,puede que lleva manana todo el dia,0.985507,0.985507,3
8,9,las casas son muy bonitas pero caras,las casas son muy bonitas pero muy cadas,0.972222,0.921053,3
25,26,el ladron al que atrapo la policia era famoso,el ladron que atrapo la policia era famoso,1.000000,0.965517,3
28,29,serias tan amable de darme el libro que esta e...,serias tan amable pa darme el libro que esta e...,0.971963,0.963636,3
29,30,hay mucha gente que no toma nada para el desayuno,hay mucha genta que no to toma nada para el de...,0.950495,0.950495,3



Rows still unscored (NA) after this pass — for next steps or manual review:


,Sentence,Stimulus_clean,Transcription_clean,token_ratio,char_ratio
4,5,que dice usted que va a hacer hoy,que dices ustedes se que van a hacer hoy,0.892308,0.904110
5,6,dudo que sepa manejar muy bien,dudo que sepa manajar bien,0.892857,0.892857
10,11,el chico con el que yo salgo es espanol,el chico con yo salgo um esta bien,0.828571,0.739726
12,13,quiero una casa en la que vivan mis animales,quiero una casa que vivan los animales,0.944444,0.878049
13,14,a nosotros nos fascinan las fiestas grandiosas,a nosotros nos fascina los fiestas grandiosos,0.945055,0.945055
15,16,me gustaria que el precio de las casas bajara,me gustaria que las precios de las casas bajaron,0.921348,0.924731
16,17,cruza a la derecha y despues sigue todo recto,cruza a la derecha y sigue a la izquierda,0.838710,0.674419
17,18,ella ha terminado de pintar su apartamento,ella ha terminado pintando su apartamento,0.915663,0.915663
18,19,me gustaria que empezara a hacer mas calor pronto,me gustaria se mas calor,0.933333,0.630137
19,20,el nino al que se le murio el gato esta triste,el nino que se murio su gato es triste,0.914286,0.857143



Counts by Score_manual after this pass:
Score_manual
3        6
4        8
<NA>    16
Name: count, dtype: int64


In [8]:
# Assign Score 2 (preserve existing Score_manual values)
# Rules:
# - Do not overwrite existing scores.
# - Assign 2 when:
#   a) token_ratio == 1.0 AND char_ratio < char_threshold_for_3 AND cleaned strings are not exactly equal
#   b) OR token_threshold_for_2 <= token_ratio < token_threshold_for_3
# - Compute token_ratio and char_ratio if missing.
# - Print rows newly assigned 2 and any remaining NA rows for manual review.

pkg = "rapidfuzz"
if importlib.util.find_spec(pkg) is None:
    subprocess.check_call([sys.executable, "-m", "-q", "install", pkg])
from rapidfuzz import fuzz

# --- Configurable thresholds (keep consistent with Score 3 cell) ---
char_threshold_for_3 = 0.95    # same threshold used in Score 3 cell
token_threshold_for_3 = 0.95   # upper bound for Score 2 (exclusive)
token_threshold_for_2 = 0.55   # lower bound for Score 2 (inclusive)
# -------------------------------

# operate on the working DataFrame
df = comparison_df.copy()

# sanity checks
if "Stimulus_clean" not in df.columns or "Transcription_clean" not in df.columns:
    raise KeyError("Expected 'Stimulus_clean' and 'Transcription_clean' in comparison_df")

# ensure Score_manual exists
if "Score_manual" not in df.columns:
    df["Score_manual"] = pd.NA

# compute token_ratio and char_ratio if missing (0-1)
if "token_ratio" not in df.columns:
    df["token_ratio"] = df.apply(
        lambda r: fuzz.token_set_ratio(str(r["Stimulus_clean"]), str(r["Transcription_clean"])) / 100.0,
        axis=1
    )
if "char_ratio" not in df.columns:
    df["char_ratio"] = df.apply(
        lambda r: fuzz.ratio(str(r["Stimulus_clean"]), str(r["Transcription_clean"])) / 100.0,
        axis=1
    )

# mask for rows still unscored (do not overwrite existing 4/3/2/1)
mask_na = df["Score_manual"].isna()

# token==1.0 but char below threshold -> assign 2 (not exact)
mask_token1_char_low = mask_na & (df["token_ratio"] == 1.0) & (df["char_ratio"] < char_threshold_for_3) & (df["Stimulus_clean"] != df["Transcription_clean"])

# token in [token_threshold_for_2, token_threshold_for_3) -> assign 2
mask_token_mid = mask_na & (df["token_ratio"] >= token_threshold_for_2) & (df["token_ratio"] < token_threshold_for_3)

# combine masks but avoid overwriting any rows that were just assigned 3 in a prior run
mask_assign_2 = (mask_token1_char_low | mask_token_mid) & df["Score_manual"].isna()

# Apply assignment
df.loc[mask_assign_2, "Score_manual"] = 2

# Reporting
print("Rows newly assigned Score_manual = 2:")
if mask_assign_2.any():
    display(df.loc[mask_assign_2, ["Sentence","Stimulus_clean","Transcription_clean","token_ratio","char_ratio","Score_manual"]])
else:
    print("(none)")

print("\nRows still unscored (NA) after this pass — manual review required (e.g., exactness check or Score 0/1):")
if df["Score_manual"].isna().any():
    display(df.loc[df["Score_manual"].isna(), ["Sentence","Stimulus_clean","Transcription_clean","token_ratio","char_ratio"]])
else:
    print("(none)")

print("\nCounts by Score_manual after this pass:")
print(df["Score_manual"].value_counts(dropna=False).sort_index())

# persist back
comparison_df = df.copy()


Rows newly assigned Score_manual = 2:


,Sentence,Stimulus_clean,Transcription_clean,token_ratio,char_ratio,Score_manual
4,5,que dice usted que va a hacer hoy,que dices ustedes se que van a hacer hoy,0.892308,0.904110,2
5,6,dudo que sepa manejar muy bien,dudo que sepa manajar bien,0.892857,0.892857,2
10,11,el chico con el que yo salgo es espanol,el chico con yo salgo um esta bien,0.828571,0.739726,2
12,13,quiero una casa en la que vivan mis animales,quiero una casa que vivan los animales,0.944444,0.878049,2
13,14,a nosotros nos fascinan las fiestas grandiosas,a nosotros nos fascina los fiestas grandiosos,0.945055,0.945055,2
15,16,me gustaria que el precio de las casas bajara,me gustaria que las precios de las casas bajaron,0.921348,0.924731,2
16,17,cruza a la derecha y despues sigue todo recto,cruza a la derecha y sigue a la izquierda,0.838710,0.674419,2
17,18,ella ha terminado de pintar su apartamento,ella ha terminado pintando su apartamento,0.915663,0.915663,2
18,19,me gustaria que empezara a hacer mas calor pronto,me gustaria se mas calor,0.933333,0.630137,2
19,20,el nino al que se le murio el gato esta triste,el nino que se murio su gato es triste,0.914286,0.857143,2



Rows still unscored (NA) after this pass — manual review required (e.g., exactness check or Score 0/1):
(none)

Counts by Score_manual after this pass:
Score_manual
2    16
3     6
4     8
Name: count, dtype: int64


In [9]:
# Assign Score 1: token_ratio < 0.55 and transcription is non-empty (preserve existing scores)

# ensure rapidfuzz is available for token_ratio if needed
pkg = "rapidfuzz"
if importlib.util.find_spec(pkg) is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])
from rapidfuzz import fuzz

df = comparison_df.copy()

# ensure cleaned columns exist
if "Stimulus_clean" not in df.columns or "Transcription_clean" not in df.columns:
    raise KeyError("Expected 'Stimulus_clean' and 'Transcription_clean' in comparison_df")

# ensure Score_manual exists
if "Score_manual" not in df.columns:
    df["Score_manual"] = pd.NA

# compute token_ratio if missing (0-1)
if "token_ratio" not in df.columns:
    df["token_ratio"] = df.apply(
        lambda r: fuzz.token_set_ratio(str(r["Stimulus_clean"]), str(r["Transcription_clean"])) / 100.0,
        axis=1
    )

# define mask: unscored, transcription non-empty, and low token similarity
mask_trans_nonempty = df["Transcription_clean"].astype(str).str.strip() != ""
mask_assign_1 = df["Score_manual"].isna() & mask_trans_nonempty & (df["token_ratio"] < 0.55)

# assign Score 1
df.loc[mask_assign_1, "Score_manual"] = 1

# show rows newly assigned 1
print("Rows newly assigned Score_manual = 1:")
if mask_assign_1.any():
    display(df.loc[mask_assign_1, ["Sentence","Stimulus_clean","Transcription_clean","token_ratio","Score_manual"]])
else:
    print("(none)")

# show rows still unscored (NA) for manual review
print("\nRows still unscored (NA) — manual review required:")
if df["Score_manual"].isna().any():
    display(df.loc[df["Score_manual"].isna(), ["Sentence","Stimulus_clean","Transcription_clean","token_ratio"]])
else:
    print("(none)")

# summary counts
print("\nCounts by Score_manual after assigning 1:")
print(df["Score_manual"].value_counts(dropna=False).sort_index())

# persist back
comparison_df = df.copy()


Rows newly assigned Score_manual = 1:
(none)

Rows still unscored (NA) — manual review required:
(none)

Counts by Score_manual after assigning 1:
Score_manual
2    16
3     6
4     8
Name: count, dtype: int64


In [10]:
# Assign Score 0: transcription_clean is empty or missing (speaker said nothing), preserve existing scores

df = comparison_df.copy()

# ensure Score_manual exists
if "Score_manual" not in df.columns:
    df["Score_manual"] = pd.NA

# define mask for "said nothing": transcription_clean is NaN or empty after stripping
mask_said_nothing = df["Transcription_clean"].isna() | (df["Transcription_clean"].astype(str).str.strip() == "")

# only assign 0 to rows still unscored and where speaker said nothing
mask_assign_0 = df["Score_manual"].isna() & mask_said_nothing
df.loc[mask_assign_0, "Score_manual"] = 0

# show rows newly assigned 0
print("Rows newly assigned Score_manual = 0 (speaker said nothing):")
if mask_assign_0.any():
    display(df.loc[mask_assign_0, ["Sentence","Stimulus_clean","Transcription_clean","token_ratio","char_ratio","Score_manual"]])
else:
    print("(none)")

# show any rows still unscored (should be none if you want to finalize everything)
print("\nRows still unscored (NA) after assigning 0 (manual review):")
if df["Score_manual"].isna().any():
    display(df.loc[df["Score_manual"].isna(), ["Sentence","Stimulus_clean","Transcription_clean","token_ratio","char_ratio"]])
else:
    print("(none)")

# final counts by score
print("\nFinal counts by Score_manual:")
print(df["Score_manual"].value_counts(dropna=False).sort_index())

# persist back
comparison_df = df.copy()


Rows newly assigned Score_manual = 0 (speaker said nothing):
(none)

Rows still unscored (NA) after assigning 0 (manual review):
(none)

Final counts by Score_manual:
Score_manual
2    16
3     6
4     8
Name: count, dtype: int64


In [11]:
# Recreate scored workbook (single-run, robust, no extra columns)

# ensure rapidfuzz is available
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "rapidfuzz"])
from rapidfuzz import fuzz

# CONFIG: set original workbook path (change if needed)
xlsx_path = "/content/drive/MyDrive/AutoEIT Test Files/Sample Audio Files and Transcriptions/AutoEIT Sample Transcriptions for Scoring.xlsx"

# output filenames (same folder)
orig = Path(xlsx_path)
out_xlsx = orig.with_name(orig.stem + "_scored_complete.xlsx")
audit_csv = orig.with_name("scoring_audit_changes.csv")

# normalization helper (same rules used previously)
def normalize_text(s, remove_parenthetical=False):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    if remove_parenthetical:
        s = re.sub(r"\s*\(\d+\)\s*$", "", s)
    s = s.lower()
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# candidate column names
stim_candidates = ["Stimulus_clean","Stimulus","Stimulus Text","Stimulus Text Clean","Stimulus_cleaned"]
trans_candidates = ["Transcription_clean","Transcription Rater 1","Transcription","Transcription_clean","Transcription_cleaned","Transcription Rater 2"]

# load all sheets
all_sheets = pd.read_excel(xlsx_path, sheet_name=None)

# prepare audit list
audit_rows = []

# process each sheet that looks like data
processed_any = False
sheet_summaries = {}

for sheet_name, df_orig in all_sheets.items():
    df = df_orig.copy()
    has_stim = any(c in df.columns for c in stim_candidates)
    has_trans = any(c in df.columns for c in trans_candidates)
    # skip non-data sheets
    if not (has_stim and has_trans and len(df.columns) >= 3):
        sheet_summaries[sheet_name] = {"processed": False, "note": "non-data sheet or too few columns"}
        continue

    # find actual column names
    stim_col = next((c for c in stim_candidates if c in df.columns), None)
    trans_col = next((c for c in trans_candidates if c in df.columns), None)
    if stim_col is None or trans_col is None:
        sheet_summaries[sheet_name] = {"processed": False, "note": "missing stim/trans columns"}
        continue

    # ensure Score_manual exists (but preserve existing non-NA)
    if "Score_manual" not in df.columns:
        df["Score_manual"] = pd.NA

    # compute cleaned text and metrics in temporary columns (never written back)
    df["_Stimulus_clean_tmp"] = df.get("Stimulus_clean", df[stim_col]).apply(lambda s: normalize_text(s, remove_parenthetical=True))
    df["_Transcription_clean_tmp"] = df.get("Transcription_clean", df[trans_col]).apply(lambda s: normalize_text(s, remove_parenthetical=False))
    df["_token_ratio_tmp"] = df.apply(lambda r: fuzz.token_set_ratio(str(r["_Stimulus_clean_tmp"]), str(r["_Transcription_clean_tmp"]))/100.0, axis=1)
    df["_char_ratio_tmp"]  = df.apply(lambda r: fuzz.ratio(str(r["_Stimulus_clean_tmp"]), str(r["_Transcription_clean_tmp"]))/100.0, axis=1)

    # apply scoring only to rows where Score_manual is NA
    mask_na_initial = df["Score_manual"].isna()

    # Score 4: exact cleaned-string equality
    mask_4 = mask_na_initial & (df["_Stimulus_clean_tmp"] == df["_Transcription_clean_tmp"])
    df.loc[mask_4, "Score_manual"] = 4

    # Score 3
    char_threshold_for_3 = 0.95
    token_threshold_for_3 = 0.95
    mask_na = df["Score_manual"].isna()
    mask_token1_char_high = mask_na & (df["_token_ratio_tmp"] == 1.0) & (df["_char_ratio_tmp"] >= char_threshold_for_3) & (df["_Stimulus_clean_tmp"] != df["_Transcription_clean_tmp"])
    mask_token_high = mask_na & (df["_token_ratio_tmp"] >= token_threshold_for_3) & (df["_token_ratio_tmp"] < 1.0) & (df["_Stimulus_clean_tmp"] != df["_Transcription_clean_tmp"])
    mask_assign_3 = mask_token1_char_high | mask_token_high
    df.loc[mask_assign_3, "Score_manual"] = 3

    # Score 2
    token_threshold_for_2 = 0.55
    mask_na = df["Score_manual"].isna()
    mask_token1_char_low = mask_na & (df["_token_ratio_tmp"] == 1.0) & (df["_char_ratio_tmp"] < char_threshold_for_3) & (df["_Stimulus_clean_tmp"] != df["_Transcription_clean_tmp"])
    mask_token_mid = mask_na & (df["_token_ratio_tmp"] >= token_threshold_for_2) & (df["_token_ratio_tmp"] < token_threshold_for_3)
    mask_assign_2 = (mask_token1_char_low | mask_token_mid) & df["Score_manual"].isna()
    df.loc[mask_assign_2, "Score_manual"] = 2

    # Score 1
    mask_na = df["Score_manual"].isna()
    mask_trans_nonempty = df["_Transcription_clean_tmp"].astype(str).str.strip() != ""
    mask_assign_1 = mask_na & mask_trans_nonempty & (df["_token_ratio_tmp"] < token_threshold_for_2)
    df.loc[mask_assign_1, "Score_manual"] = 1

    # Score 0
    mask_na = df["Score_manual"].isna()
    mask_said_nothing = df["_Transcription_clean_tmp"].isna() | (df["_Transcription_clean_tmp"].astype(str).str.strip() == "")
    mask_assign_0 = mask_na & mask_said_nothing
    df.loc[mask_assign_0, "Score_manual"] = 0

    # record audit rows for any rows that changed in this run (were NA before and now not NA)
    changed_mask = mask_4 | mask_assign_3 | mask_assign_2 | mask_assign_1 | mask_assign_0
    for idx, row in df.loc[changed_mask].iterrows():
        audit_rows.append({
            "sheet_name": sheet_name,
            "row_index": int(idx),
            "Sentence": row.get("Sentence",""),
            "Stimulus_clean": row["_Stimulus_clean_tmp"],
            "Transcription_clean": row["_Transcription_clean_tmp"],
            "token_ratio": float(row["_token_ratio_tmp"]),
            "char_ratio": float(row["_char_ratio_tmp"]),
            "Score_manual": row["Score_manual"]
        })

    # prepare final sheet to save: keep original columns order, ensure Score_manual present
    final_cols = list(df_orig.columns)
    if "Score_manual" not in final_cols:
        final_cols = final_cols + ["Score_manual"]
    final_df = df[final_cols].copy()

    # replace sheet in all_sheets with final_df
    all_sheets[sheet_name] = final_df

    # summary counts
    counts = final_df["Score_manual"].value_counts(dropna=False).sort_index().to_dict()
    sheet_summaries[sheet_name] = {"processed": True, "counts": counts}
    processed_any = True

# Save the updated workbook (overwrite if exists)
if out_xlsx.exists():
    out_xlsx.unlink()
with pd.ExcelWriter(out_xlsx, engine="openpyxl", mode="w") as writer:
    for name, sheet_df in all_sheets.items():
        sheet_df.to_excel(writer, sheet_name=name, index=False)

# Save audit CSV (all changes)
audit_df = pd.DataFrame(audit_rows)
if not audit_df.empty:
    audit_df.to_csv(audit_csv, index=False)
else:
    # write empty CSV with headers to be explicit
    pd.DataFrame(columns=["sheet_name","row_index","Sentence","Stimulus_clean","Transcription_clean","token_ratio","char_ratio","Score_manual"]).to_csv(audit_csv, index=False)

# Print concise summary
print("Scoring run complete.")
print(f"Output workbook saved as: {out_xlsx}")
print(f"Audit CSV saved as: {audit_csv}")
print("\nPer-sheet summary:")
for s, info in sheet_summaries.items():
    if not info["processed"]:
        print(f"- {s}: skipped ({info['note']})")
    else:
        print(f"- {s}: processed; counts: {info['counts']}")
print("\nOverall processed any sheets:" , processed_any)


Scoring run complete.
Output workbook saved as: /content/drive/MyDrive/AutoEIT Test Files/Sample Audio Files and Transcriptions/AutoEIT Sample Transcriptions for Scoring_scored_complete.xlsx
Audit CSV saved as: /content/drive/MyDrive/AutoEIT Test Files/Sample Audio Files and Transcriptions/scoring_audit_changes.csv

Per-sheet summary:
- Info: skipped (non-data sheet or too few columns)
- 38001-1A: processed; counts: {2: 16, 3: 6, 4: 8}
- 38002-2A: processed; counts: {1: 1, 2: 25, 3: 1, 4: 3}
- 38004-2A: processed; counts: {2: 22, 3: 4, 4: 4}
- 38006-2A: processed; counts: {1: 3, 2: 26, 3: 1}

Overall processed any sheets: True


In [12]:
p=Path("/content/drive/MyDrive/AutoEIT Test Files/Sample Audio Files and Transcriptions"); print("\n".join(sorted([f.name for f in p.iterdir()])))


038010_EIT-2A.mp3
038011_EIT-1A.mp3
AutoEIT Sample Audio for Transcribing.xlsx
AutoEIT Sample Transcriptions for Scoring.xlsx
AutoEIT Sample Transcriptions for Scoring_scored_complete.xlsx
AutoEIT Sample Transcriptions for Scoring_scored_progress.xlsx
AutoEIT Sample Transcriptions for Scoring_scored_progress_20260320_121700.xlsx
AutoEIT Sample Transcriptions for Scoring_scored_progress_20260320_121824.xlsx
AutoEIT Sample Transcriptions for Scoring_scored_progress_20260320_122649.xlsx
AutoEIT Sample Transcriptions for Scoring_scored_progress_20260320_123937.xlsx
AutoEIT Sample Transcriptions for Scoring_scored_progress_20260320_130910.xlsx
AutoEIT Sample Transcriptions for Scoring_scored_progress_20260320_131513.xlsx
AutoEIT Sample Transcriptions for Scoring_scored_progress_20260320_131514.xlsx
AutoEIT Sample Transcriptions for Scoring_scored_progress_20260322_200228.xlsx
borderline_audit.csv
scoring_audit_changes.csv


In [13]:
import pandas as pd; x="/content/drive/MyDrive/AutoEIT Test Files/Sample Audio Files and Transcriptions/AutoEIT Sample Transcriptions for Scoring_scored_complete.xlsx"; print(pd.ExcelFile(x).sheet_names)


['Info', '38001-1A', '38002-2A', '38004-2A', '38006-2A']


In [14]:
df=pd.read_excel(x, sheet_name="38001-1A"); print(df.head()); print(df["Score_manual"].value_counts(dropna=False))


   Sentence                                 Stimulus  \
0         1              Quiero cortarme el pelo (7)   
1         2             El libro está en la mesa (7)   
2         3              El carro lo tiene Pedro (8)   
3         4              El se ducha cada mañana (9)   
4         5  ¿Qué dice usted que va a hacer hoy? (9)   

                       Transcription Rater 1  Score  Score_manual  
0                    Quiero cortarme el pelo    NaN             4  
1                   El libro está en la mesa    NaN             4  
2                    El carro lo tiene Pedro    NaN             4  
3                    El se ducha cada mañana    NaN             4  
4  Que dices ustedes se que van a hacer hoy?    NaN             2  
Score_manual
2    16
4     8
3     6
Name: count, dtype: int64


In [15]:
import pandas as pd; print(pd.read_csv("/content/drive/MyDrive/AutoEIT Test Files/Sample Audio Files and Transcriptions/scoring_audit_changes.csv").head())


  sheet_name  row_index  Sentence                     Stimulus_clean  \
0   38001-1A          0         1            quiero cortarme el pelo   
1   38001-1A          1         2           el libro esta en la mesa   
2   38001-1A          2         3            el carro lo tiene pedro   
3   38001-1A          3         4            el se ducha cada manana   
4   38001-1A          4         5  que dice usted que va a hacer hoy   

                        Transcription_clean  token_ratio  char_ratio  \
0                   quiero cortarme el pelo     1.000000     1.00000   
1                  el libro esta en la mesa     1.000000     1.00000   
2                   el carro lo tiene pedro     1.000000     1.00000   
3                   el se ducha cada manana     1.000000     1.00000   
4  que dices ustedes se que van a hacer hoy     0.892308     0.90411   

   Score_manual  
0             4  
1             4  
2             4  
3             4  
4             2  


In [16]:
# cell to produce a robust borderline audit
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "rapidfuzz"])
from rapidfuzz import fuzz

# CONFIG: point to the scored workbook you confirmed
scored_xlsx = "/content/drive/MyDrive/AutoEIT Test Files/Sample Audio Files and Transcriptions/AutoEIT Sample Transcriptions for Scoring_scored_complete.xlsx"
out_csv = Path(scored_xlsx).with_name("borderline_audit.csv")

# normalizer (same rules used previously)
def normalize_text(s, remove_parenthetical=False):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    if remove_parenthetical:
        s = re.sub(r"\s*\(\d+\)\s*$", "", s)
    s = s.lower()
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# candidate column names
stim_candidates = ["Stimulus_clean","Stimulus","Stimulus Text","Stimulus Text Clean","Stimulus_cleaned"]
trans_candidates = ["Transcription_clean","Transcription Rater 1","Transcription","Transcription_clean","Transcription_cleaned","Transcription Rater 2"]

# load scored workbook
all_sheets = pd.read_excel(scored_xlsx, sheet_name=None)

borderline_rows = []
sheets_inspected = 0

for sheet_name, df_orig in all_sheets.items():
    df = df_orig.copy()
    has_stim = any(c in df.columns for c in stim_candidates)
    has_trans = any(c in df.columns for c in trans_candidates)
    if not (has_stim and has_trans):
        continue
    # only consider sheets that contain Score_manual with at least one non-NA
    if "Score_manual" not in df.columns or df["Score_manual"].isna().all():
        continue

    sheets_inspected += 1
    stim_col = next((c for c in stim_candidates if c in df.columns), None)
    trans_col = next((c for c in trans_candidates if c in df.columns), None)

    # compute cleaned text and metrics in memory
    df["Stimulus_clean"] = df.get("Stimulus_clean", df[stim_col]).apply(lambda s: normalize_text(s, remove_parenthetical=True))
    df["Transcription_clean"] = df.get("Transcription_clean", df[trans_col]).apply(lambda s: normalize_text(s, remove_parenthetical=False))
    df["token_ratio"] = df.apply(lambda r: fuzz.token_set_ratio(str(r["Stimulus_clean"]), str(r["Transcription_clean"]))/100.0, axis=1)
    df["char_ratio"]  = df.apply(lambda r: fuzz.ratio(str(r["Stimulus_clean"]), str(r["Transcription_clean"]))/100.0, axis=1)

    # borderline conditions
    cond1 = (df["token_ratio"] >= 0.90) & (df["token_ratio"] < 1.0)
    cond2 = (df["token_ratio"] == 1.0) & (df["Stimulus_clean"] != df["Transcription_clean"])
    mask = cond1 | cond2

    for idx, row in df.loc[mask].iterrows():
        borderline_rows.append({
            "sheet_name": sheet_name,
            "row_index": int(idx),
            "Sentence": row.get("Sentence",""),
            "Stimulus_clean": row["Stimulus_clean"],
            "Transcription_clean": row["Transcription_clean"],
            "token_ratio": float(row["token_ratio"]),
            "char_ratio": float(row["char_ratio"]),
            "Score_manual": row.get("Score_manual")
        })

# assemble and save
border_df = pd.DataFrame(borderline_rows)
cols = ["sheet_name","row_index","Sentence","Stimulus_clean","Transcription_clean","token_ratio","char_ratio","Score_manual"]
if border_df.empty:
    # write empty CSV with headers to be explicit
    pd.DataFrame(columns=cols).to_csv(out_csv, index=False)
else:
    border_df.to_csv(out_csv, columns=cols, index=False)

# prepare top 50 for display
if not border_df.empty:
    top50 = border_df.sort_values(["token_ratio","char_ratio"], ascending=[False,False]).head(50).copy()
    top50["token_ratio"] = top50["token_ratio"].map(lambda x: f"{x:.3f}")
    top50["char_ratio"] = top50["char_ratio"].map(lambda x: f"{x:.3f}")
else:
    top50 = pd.DataFrame(columns=cols)

# concise printed summary
print(f"Scored workbook inspected: {scored_xlsx}")
print(f"Sheets inspected (with Score_manual): {sheets_inspected}")
print(f"Total borderline rows found: {len(border_df)}")
print(f"Borderline audit CSV saved as: {out_csv}")
print("\nTop borderline rows (up to 50) for spot-check:")
display(top50)


Scored workbook inspected: /content/drive/MyDrive/AutoEIT Test Files/Sample Audio Files and Transcriptions/AutoEIT Sample Transcriptions for Scoring_scored_complete.xlsx
Sheets inspected (with Score_manual): 4
Total borderline rows found: 39
Borderline audit CSV saved as: /content/drive/MyDrive/AutoEIT Test Files/Sample Audio Files and Transcriptions/borderline_audit.csv

Top borderline rows (up to 50) for spot-check:


,sheet_name,row_index,Sentence,Stimulus_clean,Transcription_clean,token_ratio,char_ratio,Score_manual
13,38001-1A,25,26,el ladron al que atrapo la policia era famoso,el ladron que atrapo la policia era famoso,1.000,0.966,3
26,38004-2A,11,12,despues de cenar me fui a dormir tranquilo,despues de cenar fui a dormir tranquilo,1.000,0.963,3
10,38001-1A,22,23,antes de poder salir el tiene que limpiar su c...,antes de salir el tiene que limpiar su cuarto,1.000,0.938,2
21,38004-2A,2,3,el carro lo tiene pedro,el carro tiene pedro,1.000,0.930,2
36,38006-2A,1,2,el libro esta en la mesa,el libro pause esta en la mesa,1.000,0.889,2
18,38002-2A,10,11,el chico con el que yo salgo es espanol,el chico con espanol,1.000,0.678,2
38,38006-2A,28,29,serias tan amable de darme el libro que esta e...,el libro que esta en la mesa,1.000,0.675,2
1,38001-1A,7,8,puede que llueva manana todo el dia,puede que lleva manana todo el dia,0.986,0.986,3
0,38001-1A,6,7,las calles de esta ciudad son muy anchas,las calles de esta cuidad son muy anchas,0.975,0.975,3
2,38001-1A,8,9,las casas son muy bonitas pero caras,las casas son muy bonitas pero muy cadas,0.972,0.921,3
